In [1]:
# libraries for the project

import numpy as np
import gurobipy as gp
from gurobipy import GRB

import matplotlib.pyplot as plt
import json


In [5]:
# reading the json file from the test set

# we'll start easy by reading the first one

#  reading

filename = 'test01.json'

with open('test_data/'+filename, 'r') as file:
    data = json.load(file)

# creating the data structures

total_days= data['days']
total_shifts = data['shift_types']
skill_levels = data['skill_levels']
age_groups = data['age_groups']

# create a dic for occupants

occupants = data['occupants']

# create a dic for patients

patients = data['patients']

# create a dic for surgeons

surgeons = data['surgeons']

# create a dic for operating theaters

operating_theaters = data['operating_theaters']

# create a dic for rooms

rooms = data['rooms']

# saving the weights

weights = data['weights']
print(weights)

[{'id': 'a0', 'gender': 'A', 'age_group': 'elderly', 'length_of_stay': 4, 'workload_produced': [3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1], 'skill_level_required': [1, 0, 0, 0, 2, 0, 1, 1, 1, 1, 0, 0], 'room_id': 'r4'}, {'id': 'a1', 'gender': 'B', 'age_group': 'elderly', 'length_of_stay': 1, 'workload_produced': [1, 3, 1], 'skill_level_required': [1, 1, 0], 'room_id': 'r1'}, {'id': 'a2', 'gender': 'A', 'age_group': 'elderly', 'length_of_stay': 5, 'workload_produced': [2, 2, 1, 2, 2, 1, 2, 1, 1, 3, 3, 1, 1, 2, 1], 'skill_level_required': [2, 2, 0, 1, 1, 0, 0, 2, 0, 1, 0, 0, 0, 0, 0], 'room_id': 'r0'}, {'id': 'a3', 'gender': 'B', 'age_group': 'adult', 'length_of_stay': 2, 'workload_produced': [3, 2, 1, 2, 3, 1], 'skill_level_required': [2, 1, 1, 1, 1, 1], 'room_id': 'r2'}, {'id': 'a4', 'gender': 'A', 'age_group': 'elderly', 'length_of_stay': 2, 'workload_produced': [1, 1, 1, 2, 2, 1], 'skill_level_required': [1, 0, 1, 2, 0, 0], 'room_id': 'r3'}, {'id': 'a5', 'gender': 'B', 'age_group': 'elderly

In [8]:
# modifing the gender using 0 and 1

for occupant in occupants:
    if occupant['gender'] == 'A':
        occupant['gender'] = 0
    else:
        occupant['gender'] = 1

    if occupant['age_group'] == 'infant':
        occupant['age_group'] = 1                      # selected by our model
    elif occupant['age_group'] == 'adult':
        occupant['age_group'] = 2
    else:
        occupant['age_group'] = 3

print(occupants)

for patient in patients:
    if patient['gender'] == 'A':
        patient['gender'] = 0
    else:
        patient['gender'] = 1

    if patient['mandatory'] == False:
        patient['mandatory'] = 0
    else:
        patient['mandatory'] = 1

    if patient['age_group'] == 'infant':
        patient['age_group'] = 1                      # selected by our model
    elif patient['age_group'] == 'adult':
        patient['age_group'] = 2
    else:
        patient['age_group'] = 3

print(patients)

[{'id': 'a0', 'gender': 1, 'age_group': 3, 'length_of_stay': 4, 'workload_produced': [3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1], 'skill_level_required': [1, 0, 0, 0, 2, 0, 1, 1, 1, 1, 0, 0], 'room_id': 'r4'}, {'id': 'a1', 'gender': 1, 'age_group': 3, 'length_of_stay': 1, 'workload_produced': [1, 3, 1], 'skill_level_required': [1, 1, 0], 'room_id': 'r1'}, {'id': 'a2', 'gender': 1, 'age_group': 3, 'length_of_stay': 5, 'workload_produced': [2, 2, 1, 2, 2, 1, 2, 1, 1, 3, 3, 1, 1, 2, 1], 'skill_level_required': [2, 2, 0, 1, 1, 0, 0, 2, 0, 1, 0, 0, 0, 0, 0], 'room_id': 'r0'}, {'id': 'a3', 'gender': 1, 'age_group': 2, 'length_of_stay': 2, 'workload_produced': [3, 2, 1, 2, 3, 1], 'skill_level_required': [2, 1, 1, 1, 1, 1], 'room_id': 'r2'}, {'id': 'a4', 'gender': 1, 'age_group': 3, 'length_of_stay': 2, 'workload_produced': [1, 1, 1, 2, 2, 1], 'skill_level_required': [1, 0, 1, 2, 0, 0], 'room_id': 'r3'}, {'id': 'a5', 'gender': 1, 'age_group': 3, 'length_of_stay': 4, 'workload_produced': [3, 3, 1, 3, 

In [15]:
# adding a new feature to the patients, that tells us the id of the rooms where he can stay

# selecting all the rooms id

rooms_id = [room['id'] for room in rooms]

print(rooms_id)

for patient in patients:
    #(patient['incompatible_room_ids'])
    patient['compatible_rooms_ids'] = [room_id for room_id in rooms_id if room_id not in patient['incompatible_room_ids']]


print(patients)

['r0', 'r1', 'r2', 'r3', 'r4']
[{'id': 'p00', 'mandatory': 0, 'gender': 1, 'age_group': 3, 'length_of_stay': 3, 'surgery_release_day': 3, 'surgery_duration': 120, 'surgeon_id': 's0', 'incompatible_room_ids': [], 'workload_produced': [3, 2, 2, 2, 3, 1, 2, 2, 2], 'skill_level_required': [0, 2, 0, 0, 0, 0, 2, 2, 0], 'compatible_rooms_ids': ['r0', 'r1', 'r2', 'r3', 'r4']}, {'id': 'p01', 'mandatory': 0, 'gender': 1, 'age_group': 3, 'length_of_stay': 4, 'surgery_release_day': 1, 'surgery_duration': 120, 'surgeon_id': 's0', 'incompatible_room_ids': [], 'workload_produced': [3, 2, 2, 1, 1, 1, 2, 1, 1, 2, 3, 1], 'skill_level_required': [2, 1, 0, 2, 1, 0, 0, 2, 0, 2, 0, 1], 'compatible_rooms_ids': ['r0', 'r1', 'r2', 'r3', 'r4']}, {'id': 'p02', 'mandatory': 0, 'gender': 1, 'age_group': 3, 'length_of_stay': 11, 'surgery_release_day': 8, 'surgery_duration': 180, 'surgeon_id': 's0', 'incompatible_room_ids': [], 'workload_produced': [3, 2, 1, 3, 2, 1, 3, 2, 2, 2, 2, 1, 3, 2, 1, 2, 2, 1, 2, 2, 2, 2, 2